# High Stats Evaluation - Multi-File Version
This script evaluates reconstruction performance across multiple input files.
It automatically detects the number of events in each input directory.
Author: Prabhjot Singh (prabhjot@fnal.gov)
Date: 2026 March 20

In [7]:
# Case where we have multiple files with the same structure, and we want to combine the results into a single analysis.
ghosting  = True  # Set to True to include ghosting in the analysis, False to exclude ghosting
view      = "2view"  # options: "2view", "3view"
files     = "all" #"all" # 1 for 1 file, 2 for 2 files, 3 for 3 files, etc, and all for all files in the directory
events    = "all" # 1 for 1 event, 2 for 2 events, 3 for 3 events, etc, and all for all events in the file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file2", target_event = 8  (to process only file2, event 8)
# Example: target_file = "file2", target_event = None  (to process only file2, all events)
target_file  = None  # Set to "file1", "file2", etc. to process specific file only
target_event = None  # Set to event number (0, 1, 8, etc.) to process specific event only

# APA selection
apa = "APA0"  # options: "APA0", "APA1"


In [8]:
# python libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import sys
import os
from scipy.spatial import KDTree
import pandas as pd
import seaborn as sns

np.set_printoptions(linewidth=1000)

In [9]:
# Configuration: Parent directory containing multiple subdirectories
# The script will automatically find all subdirectories that contain 'xyz-coordinates'
# Each xyz-coordinates directory should contain subdirectories named with event numbers (0, 1, 2, ...)
#
# Expected structure:
# PARENT_DIR/
#   subdir1/xyz-coordinates/0/, 1/, 2/, ...
#   subdir2/xyz-coordinates/0/, 1/, 2/, ...
#   subdir3/xyz-coordinates/0/, 1/, 2/, ...

if ghosting:
    input_ghost_str = "wcp-porting-validation"
else:
    input_ghost_str = "wcp-porting-validation_without_deghosting"

PARENT_DIR = Path(f"/exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/{input_ghost_str}/sbnd/batch_results/{view}")  # CHANGE THIS to your parent directory

# Number of files to process (convert 'files' variable to num_files_to_process)
num_files_to_process = None if files == "all" else files

# Number of events to process (convert 'events' variable to num_events_to_process)
num_events_to_process = None if events == "all" else events

# Output directory for plots
if ghosting:
    output_ghost_dir = "multi_file_plots_with_deghosting"
else:
    output_ghost_dir = "multi_file_plots_without_deghosting"
PLOTBASEDIR = Path(f"/exp/sbnd/data/users/prabhjot/wirecell_clustering/cluster_evaluation/{output_ghost_dir}")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

# Parameters for selection
radius_efficiency = 1
radius_purity_xz = 1
radius_purity_yz = 2
min_recopoints_threshold = 5
min_cluster_energy = 10
min_true_points_cutoff = 200
min_reco_points_cutoff = 200
ShiftReco_Z = False
Draw_debugging_Plots = True

# Apply selections
Apply_energy_cutoff                         = True
Apply_min_true_points_cutoff                = True
Apply_min_reco_points_cutoff                = True
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False
Apply_deadarea_cut                          = True

# Fiducial volume boundaries (cm)
x_min = -250.0
x_max = 250.0
y_min = -200.0  #- 203.3
y_max = 200.0   # 200.5
z_min = 0.15    # 4.7
z_max = 500.85  # 500.6

# time window cuts
time_window_min = -205  #-1500 #-205 # in us
time_window_max = 1508.5 # 215   #1508.5 # in us

marker_size = 1

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"APA: {apa}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Time window: {time_window_min} to {time_window_max}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

# Print selective filtering info
if target_file is not None or target_event is not None:
    print(f"\n⚡ SELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")

# Print all cuts applied
print("\nCuts applied:")
if Apply_energy_cutoff:
    print(f"- Energy cutoff: {min_cluster_energy} MeV")
if Apply_min_true_points_cutoff:
    print(f"- Minimum true points cutoff: {min_true_points_cutoff}")
if Apply_min_reco_points_cutoff:
    print(f"- Minimum reco points cutoff: {min_reco_points_cutoff}")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_time_window_cut:
    print(f"- Time window cut: {time_window_min} to {time_window_max} μs")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied")

# Print cuts that are not applied
print("\nCuts not applied")
if not Apply_energy_cutoff:
    print("- Energy cutoff not applied")
if not Apply_min_true_points_cutoff:
    print("- Minimum true points cutoff not applied")
if not Apply_min_reco_points_cutoff:
    print("- Minimum reco points cutoff not applied")
if not Apply_wire_readout_sensitive_xz_plane_cut:
    print("- Wire readout sensitive xz plane cut not applied")
if not Apply_time_window_cut:
    print("- Time window cut not applied")
if not Apply_deadarea_cut:
    print("- Dead area cut not applied")

Configuration:
Parent directory: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view
APA: APA0
Plot base directory: /exp/sbnd/data/users/prabhjot/wirecell_clustering/cluster_evaluation/multi_file_plots_with_deghosting
Time window: -205 to 1508.5
Files to process: all
Events to process: all

Cuts applied:
- Energy cutoff: 10 MeV
- Minimum true points cutoff: 200
- Minimum reco points cutoff: 200
- Wire readout sensitive xz plane cut applied
- Dead area cut applied

Cuts not applied
- Time window cut not applied


In [10]:
# Import functions from Python modules
from efficiency_purity_estimate import EvaluateEfficiency, EvaluatePurity
from efficiency_purity_draw import (
    plot_efficiency_heatmap, plot_purity_heatmap,
    DrawEfficiencyVsTrueEnergyPerEvent, DrawEfficiencyVsTrueEnergyPerFile, DrawEfficiencyVsTrueEnergyPerJob,
    DrawPurityVsRecoChargePerEvent,
    DrawAggregatedEfficiencyPlots, DrawAggregatedPurityPlots
)
from readfiles import read_files_for_event
from selections import (
    apply_energy_cutoff, apply_min_true_points_cutoff, apply_min_reco_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true, apply_wire_readout_sensitive_yz_plane_cut_reco,
    reassign_cluster_ID_true, reassign_cluster_ID_reco, GroupClustersByID, ShiftRecoClusterZValues, apply_time_window_cut,
    apply_deadarea_cut_true
)
from DrawRecoTrueClusters import DrawTrueRecoClustersXZ, DrawTrueRecoClustersYZ, DrawTrueRecoClustersXY, DrawTrueClusterWithMatchedReco, DrawLabels
from clusterpairmatching import MatchTruetoReco_OneToMany
from bee_display_link import print_bee_display_link
from cluster_category import cluster_category
from metadata import add_metadata_true_clusters, aggregate_metadata, print_metadata

In [ ]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file1/, file2/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs

def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"
    
    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []
    
    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                event_num = int(item.name)
                events.append(event_num)
            except ValueError:
                pass
    
    return sorted(events)

# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
# Limit to num_files_to_process files for testing
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        events = detect_events_in_directory(input_dir)
        if events:
            print(f"  {input_dir.name}/data/: {len(events)} events ({min(events)}-{max(events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")
    print("    Please check that:")
    print("    1. PARENT_DIR is set correctly")
    print("    2. Subdirectories (file1/, file2/, etc.) contain 'data/' folders")
    print("    3. data/ folders contain event directories (0, 1, 2, ...)")

Scanning parent directory: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view
------------------------------------------------------------
Found: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file1
Found: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file10
Found: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file2
Found: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file3
Found: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file4
Found: /exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file5
Found: /exp/sbnd/data/users/prabhjot/wirecell_clustering/develo

: 

In [ ]:
# ============================================================================
# RESTRUCTURED MAIN PROCESSING LOOP - HIERARCHICAL EVENT/FILE/JOB ANALYSIS
# ============================================================================
# Structure:
#   Job Level
#     ├─ File Level Loop
#     │   ├─ Event Level Loop
#     │   │   ├─ Efficiency/Purity Heatmaps (all clusters, no pairing)
#     │   │   ├─ True Cluster Visualizations (with matched reco clusters)
#     │   │   ├─ Efficiency vs True Energy plots (per true cluster)
#     │   │   └─ Purity vs Reco Charge plots (per reco cluster)
#     │   ├─ Event-Level Aggregation (2D/1D plots for all clusters in event)
#     │   └─ File-Level Aggregation (2D/1D plots for all clusters in file)
#     └─ Job-Level Aggregation (2D/1D plots across all files)
# ============================================================================

if not input_directories:
    print("No input directories found. Cannot proceed with processing.")
else:
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    output_dir_view = PLOTBASEDIR / f"{view}"
    output_dir      = output_dir_view / f"apa_{apa}_{timestamp}"
    
    # Create output directory
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*70}")
    print(f"📁 Output directory created with timestamp:")
    print(f"   {output_dir}")
    print(f"{'='*70}\n")
    
    # Aggregate containers for each level
    job_efficiency_results  = []      # All efficiency results across all files/events
    job_purity_results      = []      # All purity results across all files/events
    job_matched_pairs       = []      # Matched pairs across all files/events
    input_directories_map   = {}      # Track event -> (input_dir, evt_num) mapping
    job_bee_links           = []      # Collect bee display links for all files
    job_metadata_list       = []      # Metadata for all true clusters across all files/events
    
    total_events_processed  = 0
    total_files_processed   = 0

    # ========================================================================
    # FILE LEVEL LOOP
    # ========================================================================
    for file_idx, input_dir in enumerate(input_directories):
        input_file_name = input_dir.name
        
        # SELECTIVE FILTERING: Skip files that don't match target_file
        if target_file is not None and input_file_name != target_file:
            print(f"Skipping {input_file_name} (target: {target_file})")
            continue
        
        print(f"\n{'='*70}")
        # in following also print exact name of the input directory being processed
        print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
        
        print(f"{'='*70}")
        
        file_output_dir = output_dir / input_file_name
        file_output_dir.mkdir(parents=True, exist_ok=True)
        
        # Containers for file-level aggregation
        file_efficiency_results = []
        file_purity_results     = []
        file_matched_pairs      = []
        file_metadata_list      = []      # Metadata for all true clusters in this file
        
        # Get event range for this directory
        events_list = detect_events_in_directory(input_dir)
        if not events_list:
            print(f"No events found in {input_dir}, skipping...")
            continue
        
        event_low  = min(events_list)
        # Determine event_high based on num_events_to_process
        if num_events_to_process is None:
            event_high = max(events_list) + 1  # Process all events
        else:
            event_high = event_low + num_events_to_process  # Process specified number of events

        bee_url = print_bee_display_link(input_dir)
        if bee_url:
            job_bee_links.append({'file': input_file_name, 'url': bee_url})
        print(f"Processing events {event_low} to {event_high-1}\n")
        
        # ====================================================================
        # EVENT LEVEL LOOP
        # ====================================================================
        for evt in range(event_low, event_high):
            # SELECTIVE FILTERING: Skip events that don't match target_event
            if target_event is not None and evt != target_event:
                continue
            
            print(f"  {'='*60}")
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            print(f"  EVENT {evt}")
            print(f"  {'='*60}")
            
            result = read_files_for_event(input_dir, evt, apa)
            if result is None:
                print(f"  Could not read data, skipping event {evt}")
                continue
            
            # Setup event output directories
            event_key                        = f"{input_file_name}_{evt}"
            input_directories_map[event_key] = (input_dir, evt)
            
            event_output_dir = file_output_dir / f"event_{evt:03d}"
            event_output_dir.mkdir(parents=True, exist_ok=True)
            PLOTDIR_EVT     = event_output_dir
            
            efficiency_dir  = event_output_dir / "efficiency"
            purity_dir      = event_output_dir / "purity"
            efficiency_dir  .mkdir(parents=True, exist_ok=True)
            purity_dir      .mkdir(parents=True, exist_ok=True)
            
            # Load and process data
            x_true, y_true, z_true, id_true, q_true, e_true, t_true, x_pred, y_pred, z_pred, id_pred, q_pred = result
            
            true_5d_points              = np.column_stack((x_true, y_true, z_true, id_true, q_true, e_true, t_true))
            true_5d_points              = reassign_cluster_ID_true(true_5d_points)
            true_5d_points_unfiltered   = true_5d_points.copy()  # Keep an unfiltered copy for later comparison

            # Apply selections to true and reco points
            # True points here
            if Apply_energy_cutoff:
                true_5d_points = apply_energy_cutoff(true_5d_points, min_cluster_energy)
            if Apply_min_true_points_cutoff:
                true_5d_points = apply_min_true_points_cutoff(true_5d_points, min_true_points_cutoff)
            if Apply_wire_readout_sensitive_xz_plane_cut:
                true_5d_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_5d_points, x_min, x_max, y_min, y_max, z_min, z_max)
            if Apply_deadarea_cut:
                true_5d_points = apply_deadarea_cut_true(true_5d_points, apa, view_type=view, output_dir=PLOTDIR_EVT, event=evt, file_name=input_file_name)
            if Apply_time_window_cut:
                true_5d_points = apply_time_window_cut(true_5d_points, time_window_min, time_window_max, apa)

            # Reco points here
            clusters_true            = GroupClustersByID(true_5d_points)
            clusters_true_unfiltered = GroupClustersByID(true_5d_points_unfiltered)
            
            predicted_5d_points = np.column_stack((x_pred, y_pred, z_pred, id_pred, q_pred))
            if Apply_min_reco_points_cutoff:
                predicted_5d_points = apply_min_reco_points_cutoff(predicted_5d_points, min_reco_points_cutoff)
            if Apply_wire_readout_sensitive_xz_plane_cut:
                predicted_5d_points = apply_wire_readout_sensitive_yz_plane_cut_reco(predicted_5d_points, x_min, x_max, y_min, y_max, z_min, z_max)

            # All selections applied above

            # Reassign cluster IDs and group into clusters after all selections are applied
            predicted_5d_points = reassign_cluster_ID_reco(predicted_5d_points)
            clusters_reco       = GroupClustersByID(predicted_5d_points)
            
            if ShiftReco_Z:
                clusters_reco = ShiftRecoClusterZValues(clusters_reco, shift_value=0.5)

            # Call function to draw the true and the reco clusters together in the same plot to visually compare them
            # We will use different colors for true and reco clusters.
            from DrawRecoTrueClusters import DrawTrueRecoClustersXZ, DrawTrueRecoClustersYZ, DrawTrueRecoClustersXY
            DrawTrueRecoClustersXZ(clusters_true, clusters_reco, evt, apa, PLOTDIR_EVT, input_file_name)
            DrawTrueRecoClustersYZ(clusters_true, clusters_reco, evt, apa, PLOTDIR_EVT, input_file_name)
            DrawTrueRecoClustersXY(clusters_true, clusters_reco, evt, apa, PLOTDIR_EVT, input_file_name)
            DrawLabels(clusters_true, evt, apa, PLOTDIR_EVT, input_file_name)

            # Analyze cluster categories based on angle in XZ plane
            print(f"    Analyzing cluster categories for {len(clusters_true)} true clusters...")
            cluster_category_results = cluster_category(clusters_true, output_dir=PLOTDIR_EVT, event=evt, apa=apa, file_name=input_file_name)
            print(f"    Cluster category analysis complete: {len(cluster_category_results)} clusters analyzed")

            # ================================================================
            # EVALUATE EFFICIENCY AND PURITY
            # ================================================================
            from efficiency_purity_estimate import EvaluateEfficiency, EvaluatePurity
            efficiency_results  = EvaluateEfficiency(clusters_true, clusters_reco, event_key, radius_efficiency, min_recopoints_threshold)
            job_efficiency_results.extend(efficiency_results)

            purity_results      = EvaluatePurity(clusters_true, clusters_reco, event_key, radius_purity_xz, radius_purity_yz)
            job_purity_results.extend(purity_results)
            
            # ================================================================
            # COLLECT CLUSTER METADATA
            # ================================================================
            print(f"    Collecting cluster metadata for {len(clusters_true)} true clusters...")
            event_metadata_list = add_metadata_true_clusters(
                efficiency_results,
                cluster_category_results,
                file_name=input_file_name,
                event=evt,
                apa=apa,
                view=view,
                event_key=event_key
            )
            print(f"    Metadata collected for {len(event_metadata_list)} clusters")
            
            # Display event-level metadata
            if event_metadata_list:
                print_metadata(event_metadata_list)
            
            # Aggregate to file and job levels
            file_metadata_list.extend(event_metadata_list)
            job_metadata_list.extend(event_metadata_list)
            
            # ================================================================
            # EVENT-LEVEL PROCESSING: HEATMAPS AND TRUE CLUSTER VISUALIZATIONS
            # ================================================================
            print(f"    [1/5] Drawing efficiency/purity heatmaps...")
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            print(f"  EVENT {evt}")
            from efficiency_purity_draw import plot_efficiency_heatmap, plot_purity_heatmap
            plot_efficiency_heatmap(efficiency_results, evt, apa, efficiency_dir, input_file_name)
            plot_purity_heatmap(purity_results, evt, apa, purity_dir, input_file_name)
            
            print(f"    [2/5] Drawing true clusters with matched reco clusters...")
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            print(f"  EVENT {evt}")
            matched_true_reco_clusters = MatchTruetoReco_OneToMany(purity_results, efficiency_results)
            for matched_info in matched_true_reco_clusters:
                DrawTrueClusterWithMatchedReco(matched_info, clusters_true, clusters_reco, efficiency_dir, evt, apa, input_file_name)
            
            print(f"    [3/5] Drawing efficiency vs true cluster energy plots...")
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            print(f"  EVENT {evt}")
            DrawEfficiencyVsTrueEnergyPerEvent(efficiency_results, efficiency_dir, evt, apa, input_file_name, cluster_category_results=cluster_category_results)
            
            print(f"    [4/5] Drawing purity vs reco charge plots...")
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            print(f"  EVENT {evt}")
            DrawPurityVsRecoChargePerEvent(purity_results, purity_dir, evt, apa, input_file_name)
            
            # Aggregate results
            file_efficiency_results.extend(efficiency_results)
            file_purity_results.extend(purity_results)
            
            total_events_processed += 1
            print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
            print(f"  EVENT {evt}")
            print(f"    [5/5] Event {evt} complete: {len(clusters_true)} true, {len(clusters_reco)} reco clusters\n")
        
        # ====================================================================
        # FILE-LEVEL AGGREGATION (After all events in file are processed)
        # ====================================================================
        print(f"\n{'='*70}")
        print(f"FILE-LEVEL AGGREGATION: Generating file-level summary plots...")
        print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir.parent.name} ({input_dir})")
        print(f"{'='*70}")
        print(f"Total events processed in file: {total_events_processed}")
        print(f"Total efficiency results in file: {len(file_efficiency_results)}")
        print(f"Total purity results in file: {len(file_purity_results)}")
        print(f"Total matched pairs in file: {len(file_matched_pairs)}")
        if file_efficiency_results and file_purity_results:
            print(f"\n  FILE-LEVEL AGGREGATION: Generating file-level summary plots...")
            print(f"  Total clusters in file: {len(file_efficiency_results)} efficiency, {len(file_purity_results)} purity")
            
            agg_output_dir = file_output_dir / "file_summary"
            agg_output_dir.mkdir(parents=True, exist_ok=True)

            DrawEfficiencyVsTrueEnergyPerFile(file_efficiency_results, agg_output_dir, apa, input_file_name, file_metadata_list=file_metadata_list)
            DrawPurityVsRecoChargePerEvent(file_purity_results, agg_output_dir, evt, apa, input_file_name)

            #DrawAggregatedEfficiencyPlots(file_efficiency_results, agg_output_dir, "File-Level", apa)
            #DrawAggregatedPurityPlots(file_purity_results, agg_output_dir, "File-Level", apa)
        
        # ================================================================
        # FILE-LEVEL METADATA SUMMARY
        # ================================================================
        if file_metadata_list:
            file_metadata_stats = aggregate_metadata(file_metadata_list)
            #print(f"\n  FILE-LEVEL METADATA SUMMARY ({input_file_name}):")
            #print(f"  Total clusters: {file_metadata_stats['total_clusters']}")
            #print(f"  By type - Neutrino: {file_metadata_stats['by_type']['neutrino']}, Cosmic: {file_metadata_stats['by_type']['cosmic']}")
            #print(f"  By category - Isochronous: {file_metadata_stats['by_category']['isochronous']}, Prolonged: {file_metadata_stats['by_category']['prolonged']}, Normal: {file_metadata_stats['by_category']['normal']}")
            #print(f"  Efficiency - Mean: {file_metadata_stats['efficiency_stats']['mean']:.4f}, Median: {file_metadata_stats['efficiency_stats']['median']:.4f}")
            #print(f"  Reco Matches - Mean: {file_metadata_stats['reco_matches_stats']['mean']:.2f}, Median: {file_metadata_stats['reco_matches_stats']['median']:.2f}\n")

        # Aggregate to job level
        #job_efficiency_results.extend(file_efficiency_results)
        #job_purity_results.extend(file_purity_results)
        #job_matched_pairs.extend(file_matched_pairs)
        total_files_processed += 1
    
    # ========================================================================
    # JOB-LEVEL AGGREGATION (After all files are processed)
    # ========================================================================
    print(f"\n{'='*70}")
    print(f"JOB-LEVEL AGGREGATION: Generating job-level summary plots...")
    print(f"{'='*70}")
    print(f"Total files processed: {total_files_processed}")
    print(f"Total events processed: {total_events_processed}")
    print(f"Total efficiency results: {len(job_efficiency_results)}")
    print(f"Total purity results: {len(job_purity_results)}")
    print(f"Total matched pairs: {len(job_matched_pairs)}")
    
    if job_efficiency_results and job_purity_results:
        job_agg_output_dir = output_dir / "job_summary"
        job_agg_output_dir.mkdir(parents=True, exist_ok=True)

        DrawEfficiencyVsTrueEnergyPerJob(job_efficiency_results, job_agg_output_dir, apa, job_metadata_list=job_metadata_list)
        DrawPurityVsRecoChargePerEvent(job_purity_results, job_agg_output_dir, evt, apa)
        
        #DrawAggregatedEfficiencyPlots(job_efficiency_results, job_agg_output_dir, "Job-Level", apa)
        #DrawAggregatedPurityPlots(job_purity_results, job_agg_output_dir, "Job-Level", apa)
        if job_matched_pairs:
            DrawMatchedPairsPlots(job_matched_pairs, job_agg_output_dir, "Job-Level", apa)
        
        print(f"\nJob-level summary statistics:")
        print(f"  Mean Efficiency: {np.mean([e['efficiency_energy_weighted'] for e in job_efficiency_results]):.4f}")
        print(f"  Mean Purity: {np.mean([p['purity'] for p in job_purity_results]):.4f}")
    
    # ================================================================
    # JOB-LEVEL METADATA SUMMARY
    # ================================================================
    if job_metadata_list:
        job_metadata_stats = aggregate_metadata(job_metadata_list)

        Print_Metadata = False  # Set to True to print job-level metadata summary

        if Print_Metadata:
            print(f"\n{'='*70}")
            print(f"JOB-LEVEL METADATA SUMMARY:")
            print(f"{'='*70}")
            print(f"Total clusters across all files: {job_metadata_stats['total_clusters']}")
            print(f"\nBy type:")
            print(f"  Neutrino: {job_metadata_stats['by_type']['neutrino']}")
            print(f"  Cosmic: {job_metadata_stats['by_type']['cosmic']}")
            print(f"\nBy category:")
            print(f"  Isochronous: {job_metadata_stats['by_category']['isochronous']}")
            print(f"  Prolonged: {job_metadata_stats['by_category']['prolonged']}")
            print(f"  Normal: {job_metadata_stats['by_category']['normal']}")
            print(f"\nEfficiency Statistics:")
            print(f"  Mean: {job_metadata_stats['efficiency_stats']['mean']:.4f}")
            print(f"  Median: {job_metadata_stats['efficiency_stats']['median']:.4f}")
            print(f"  Min: {job_metadata_stats['efficiency_stats']['min']:.4f}")
            print(f"  Max: {job_metadata_stats['efficiency_stats']['max']:.4f}")
            print(f"\nReco Matches Statistics:")
            print(f"  Mean: {job_metadata_stats['reco_matches_stats']['mean']:.2f}")
            print(f"  Median: {job_metadata_stats['reco_matches_stats']['median']:.2f}")
            print(f"  Min: {job_metadata_stats['reco_matches_stats']['min']}")
            print(f"  Max: {job_metadata_stats['reco_matches_stats']['max']}")
            print(f"{'='*70}\n")
    
    # ========================================================================
    # SAVE BEE DISPLAY LINKS TO FILE
    # ========================================================================
    if job_bee_links:
        bee_links_file = output_dir / "job_summary" / "bee_display_links.txt"
        with open(bee_links_file, 'w') as f:
            f.write("BEE DISPLAY LINKS FOR ALL FILES\n")
            f.write("=" * 80 + "\n\n")
            for link_info in job_bee_links:
                f.write(f"{link_info['file']}:\n")
                f.write(f"  {link_info['url']}\n\n")
        print(f"\nBee display links saved to: {bee_links_file}")
    else:
        print("\nNo bee display links to save")


📁 Output directory created with timestamp:
   /exp/sbnd/data/users/prabhjot/wirecell_clustering/cluster_evaluation/multi_file_plots_with_deghosting/2view/apa_APA0_20260709_170204


FILE 1/10: 2view (/exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file1)

BEE-DISPLAY LINKS for file1
  https://www.phy.bnl.gov/twister/bee/set/ac70db88-68d4-4b24-a141-ea66c8eafe83/event/list/

Processing events 0 to 7

FILE 1/10: 2view (/exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file1)
  EVENT 0
Applying Dead Area Cut
APA: APA0, View: 2view
Number of points before dead area cut: 41360
Number of points after dead area cut: 41360
Points removed: 0 (0.0%)

Drew before/after dead area visualizations
    Analyzing cluster categories for 7 true clusters...

    Cluster category analysis complete: 7 clusters analyzed
    Metadata collected for 7 clusters

File       Event    APA    View

/exp/sbnd/data/users/prabhjot/wirecell_clustering/cluster_evaluation/DrawRecoTrueClusters.py:481: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  patch = MplPolygon(polygon_swapped, closed=True, alpha=0.6, color='black',


  Drew cluster 9999 (full cluster with dead area overlay)
    Analyzing cluster categories for 6 true clusters...

    Cluster category analysis complete: 6 clusters analyzed
    Metadata collected for 6 clusters

File       Event    APA    View     Cluster ID   Type       Category        Efficiency   Reco Matches   
file1      file1_2  APA0   2view    9999         neutrino   normal          0.4966       1              
file1      file1_2  APA0   2view    19           cosmic     normal          0.9488       1              
file1      file1_2  APA0   2view    -142         cosmic     isochronous     0.4804       1              
file1      file1_2  APA0   2view    -11          cosmic     normal          0.9884       1              
file1      file1_2  APA0   2view    -220         cosmic     isochronous     0.2276       1              
file1      file1_2  APA0   2view    -78          cosmic     normal          0.5477       2              

    [1/5] Drawing efficiency/purity heatmaps...
FI

/exp/sbnd/data/users/prabhjot/wirecell_clustering/cluster_evaluation/DrawRecoTrueClusters.py:481: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  patch = MplPolygon(polygon_swapped, closed=True, alpha=0.6, color='black',


  Drew cluster -33 (full cluster with dead area overlay)
    Analyzing cluster categories for 11 true clusters...

    Cluster category analysis complete: 11 clusters analyzed
    Metadata collected for 11 clusters

File       Event    APA    View     Cluster ID   Type       Category        Efficiency   Reco Matches   
file1      file1_4  APA0   2view    -129         cosmic     isochronous     0.9987       1              
file1      file1_4  APA0   2view    -181         cosmic     normal          0.6385       1              
file1      file1_4  APA0   2view    -33          cosmic     normal          0.8973       1              
file1      file1_4  APA0   2view    -54          cosmic     normal          0.9290       1              
file1      file1_4  APA0   2view    -52          cosmic     prolonged       0.9906       1              
file1      file1_4  APA0   2view    -13          cosmic     prolonged       0.9908       1              
file1      file1_4  APA0   2view    -166         

...

  Drew cluster -1 (full cluster with dead area overlay)
    Analyzing cluster categories for 4 true clusters...

    Cluster category analysis complete: 4 clusters analyzed
    Metadata collected for 4 clusters

File       Event    APA    View     Cluster ID   Type       Category        Efficiency   Reco Matches   
file3      file3_1  APA0   2view    9999         neutrino   normal          0.7018       1              
file3      file3_1  APA0   2view    -157         cosmic     isochronous     0.9269       1              
file3      file3_1  APA0   2view    -2           cosmic     normal          0.9123       1              
file3      file3_1  APA0   2view    -207         cosmic     normal          0.0000       1              

    [1/5] Drawing efficiency/purity heatmaps...
FILE 4/10: 2view (/exp/sbnd/data/users/prabhjot/wirecell_clustering/developcode/wcp-porting-validation/sbnd/batch_results/2view/file3)
  EVENT 1
    [2/5] Drawing true clusters with matched reco clusters...
FILE 4/

/exp/sbnd/data/users/prabhjot/wirecell_clustering/cluster_evaluation/DrawRecoTrueClusters.py:481: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  patch = MplPolygon(polygon_swapped, closed=True, alpha=0.6, color='black',


  Drew cluster -101 (full cluster with dead area overlay)
    Analyzing cluster categories for 12 true clusters...

    Cluster category analysis complete: 12 clusters analyzed
    Metadata collected for 12 clusters

File       Event    APA    View     Cluster ID   Type       Category        Efficiency   Reco Matches   
file3      file3_2  APA0   2view    -171         cosmic     isochronous     0.7808       1              
file3      file3_2  APA0   2view    -0           cosmic     isochronous     0.9853       1              
file3      file3_2  APA0   2view    -101         cosmic     normal          0.0536       1              
file3      file3_2  APA0   2view    2            cosmic     normal          0.9718       1              
file3      file3_2  APA0   2view    -126         cosmic     isochronous     0.9727       1              
file3      file3_2  APA0   2view    23           cosmic     normal          0.9799       1              
file3      file3_2  APA0   2view    -197        

/exp/sbnd/data/users/prabhjot/wirecell_clustering/cluster_evaluation/DrawRecoTrueClusters.py:481: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  patch = MplPolygon(polygon_swapped, closed=True, alpha=0.6, color='black',


  Drew cluster -123 (full cluster with dead area overlay)
    Analyzing cluster categories for 7 true clusters...

    Cluster category analysis complete: 7 clusters analyzed
    Metadata collected for 7 clusters

File       Event    APA    View     Cluster ID   Type       Category        Efficiency   Reco Matches   
file3      file3_3  APA0   2view    -230         cosmic     isochronous     0.8776       1              
file3      file3_3  APA0   2view    -44          cosmic     normal          0.7616       1              
file3      file3_3  APA0   2view    -124         cosmic     normal          0.9689       1              
file3      file3_3  APA0   2view    -165         cosmic     prolonged       0.4321       1              
file3      file3_3  APA0   2view    28           cosmic     normal          0.2295       1              
file3      file3_3  APA0   2view    30           cosmic     normal          0.0000       1              
file3      file3_3  APA0   2view    -152         co